# Lab 4 — Tensor Products, Phase and No-Cloning

**Learning objectives**

- Construct product states
- Distinguish global and relative phase
- Explain why CNOT cannot clone arbitrary states

## Environment setup

Execute Below

In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## Tensor products

Tensor products combine subsystem states. Qiskit uses right-to-left qubit ordering.

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector, state_fidelity
zero=Statevector.from_label("0"); one=Statevector.from_label("1")
plus=Statevector([1,1])/np.sqrt(2)
for name,s in {"|0>⊗|1>":zero.tensor(one),"|+>⊗|0>":plus.tensor(zero)}.items():
    print(name); display(s.draw("latex"))

## Global phase

Multiplying the whole state by a phase leaves probabilities unchanged.

In [ ]:
psi=plus; phased=np.exp(1j*np.pi/3)*psi
print(psi.probabilities_dict(), phased.probabilities_dict(), "fidelity=",state_fidelity(psi,phased))

## Relative phase

$|+
angle$ and $|−
angle$ have identical Z-basis probabilities, but H reveals the difference.

In [ ]:
from qiskit import QuantumCircuit
plus=Statevector([1,1])/np.sqrt(2); minus=Statevector([1,-1])/np.sqrt(2)
print("Before H:",plus.probabilities_dict(),minus.probabilities_dict())
print("After H:",plus.evolve(Operator(QuantumCircuit(1).compose(QuantumCircuit(1)))).probabilities_dict() if False else "shown below")
h=QuantumCircuit(1); h.h(0)
print(plus.evolve(h).probabilities_dict(),minus.evolve(h).probabilities_dict())

## No-cloning experiment

CNOT copies basis labels but turns $|+
angle|0
angle$ into an entangled state—not $|+
angle|+
angle$.

In [ ]:
from qiskit.quantum_info import partial_trace
attempt=QuantumCircuit(2); attempt.h(0); attempt.cx(0,1)
actual=Statevector.from_instruction(attempt); desired=plus.tensor(plus)
display(actual.draw("latex")); display(desired.draw("latex"))
print("Fidelity with two copies:",state_fidelity(actual,desired))
print("Reduced q0 state:\n",partial_trace(actual,[1]).data)